# CIFAR-10 cSGHMC Training + Ensemble Evaluation

This notebook version is adapted from the original single-file script.

Main changes for Jupyter:

- No `argparse`.
- No `__file__`.
- Configuration is done in the first code cell.
- The `models/` directory is added manually based on the assumed project structure:

```text
project/
├── experiments/
│   └── this_notebook.ipynb
└── models/
    └── resnet.py
```

In [ ]:
# Configuration

import os
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# If this notebook is inside project/experiments/
# and resnet.py is inside project/models/resnet.py,
# then this path should work.
MODELS_DIR = Path.cwd().parent / "models"
sys.path.append(str(MODELS_DIR))

from resnet import ResNet18

DATASET_SIZE = 50000
NUM_CYCLES = 4
INITIAL_LR = 0.5
WEIGHT_DECAY = 5e-4
CHECKPOINT_PREFIX = "cifar_csghmc"

# Edit these values for your experiment.
RUN_DIR = "./checkpoints"
DATA_PATH = "./data"

EPOCHS = 200
BATCH_SIZE = 64
ALPHA = 0.9
DEVICE_ID = 0
SEED = 1
TEMPERATURE = 1.0 / DATASET_SIZE

# Set to None to use all saved checkpoints.
EVAL_NUM_MODELS = None

## Reproducibility and device setup

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def get_device(device_id=0):
    if torch.cuda.is_available():
        cudnn.benchmark = True
        cudnn.deterministic = True
        return torch.device(f"cuda:{device_id}")
    return torch.device("cpu")


os.makedirs(RUN_DIR, exist_ok=True)

set_seed(SEED)
device = get_device(DEVICE_ID)

print("Using device:", device)
print("Saving checkpoints to:", RUN_DIR)
print("Using data path:", DATA_PATH)

## Build CIFAR-10 dataloaders

In [ ]:
def build_dataloaders(data_path, batch_size):
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_set = torchvision.datasets.CIFAR10(
        root=data_path,
        train=True,
        download=True,
        transform=train_transform,
    )

    test_set = torchvision.datasets.CIFAR10(
        root=data_path,
        train=False,
        download=True,
        transform=test_transform,
    )

    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    test_loader = torch.utils.data.DataLoader(
        test_set,
        batch_size=100,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, test_loader


train_loader, test_loader = build_dataloaders(DATA_PATH, BATCH_SIZE)
num_batches = len(train_loader)

print("Training batches:", num_batches)
print("Test batches:", len(test_loader))

## Learning-rate schedule

The step size / learning rate follows a cosine cycle:

\[
\eta_t =
\frac{1}{2}
\left(
\cos\left(
\frac{\pi (t \bmod L)}{L}
\right)
+ 1
\right)
\eta_0
\]

where:

- \(t\) is the current mini-batch iteration,
- \(L\) is the length of one cycle,
- \(\eta_0\) is the initial learning rate.

In [ ]:
def cyclical_lr(epoch, batch_idx, num_batches, total_epochs):
    total_steps = total_epochs * num_batches
    step = epoch * num_batches + batch_idx
    cycle_length = total_steps // NUM_CYCLES

    cos_inner = np.pi * (step % cycle_length) / cycle_length
    lr = 0.5 * (np.cos(cos_inner) + 1.0) * INITIAL_LR

    return lr

## Custom cSGHMC-style parameter update

In [ ]:
def update_params(model, lr, epoch, alpha, temperature, device):
    for param in model.parameters():
        if param.grad is None:
            continue

        if not hasattr(param, "buf"):
            param.buf = torch.zeros_like(param, device=device)

        grad = param.grad.data
        grad.add_(param.data, alpha=WEIGHT_DECAY)

        buf_new = (1.0 - alpha) * param.buf - lr * grad

        # Noise is injected during epochs 46-50 of each 50-epoch cycle.
        if (epoch % 50) + 1 > 45:
            eps = torch.randn_like(param, device=device)
            noise_scale = (2.0 * lr * alpha * temperature / DATASET_SIZE) ** 0.5
            buf_new = buf_new + noise_scale * eps

        param.data.add_(buf_new)
        param.buf = buf_new

## Training and evaluation functions

In [ ]:
def train_one_epoch(model, loader, criterion, device, epoch):
    model.train()

    running_loss = 0.0
    running_correct = 0
    seen = 0

    print(f"\nEpoch: {epoch}")

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        model.zero_grad()

        lr = cyclical_lr(epoch, batch_idx, num_batches, EPOCHS)

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()
        update_params(model, lr, epoch, ALPHA, TEMPERATURE, device)

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)

        seen += targets.size(0)
        running_correct += predicted.eq(targets).sum().item()

        if batch_idx % 100 == 0:
            print(
                "Loss: %.3f | Acc: %.3f%% (%d/%d)"
                % (
                    running_loss / (batch_idx + 1),
                    100.0 * running_correct / seen,
                    running_correct,
                    seen,
                )
            )


def evaluate_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(loader):
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total_seen += targets.size(0)
            total_correct += predicted.eq(targets).sum().item()

            if batch_idx % 100 == 0:
                print(
                    "Test Loss: %.3f | Test Acc: %.3f%% (%d/%d)"
                    % (
                        total_loss / (batch_idx + 1),
                        100.0 * total_correct / total_seen,
                        total_correct,
                        total_seen,
                    )
                )

    print(
        "\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n".format(
            total_loss / len(loader),
            total_correct,
            total_seen,
            100.0 * total_correct / total_seen,
        )
    )

## Create the model

In [ ]:
model = ResNet18().to(device)
criterion = nn.CrossEntropyLoss()

print(model.__class__.__name__)

## Train and save checkpoints

With the default `EPOCHS = 200`, this saves 3 checkpoints at the end of each 50-epoch cycle, for 12 checkpoints total.

In [ ]:
checkpoint_idx = 0

for epoch in range(EPOCHS):
    train_one_epoch(model, train_loader, criterion, device, epoch)
    evaluate_epoch(model, test_loader, criterion, device)

    # Save checkpoints during epochs 48, 49, and 50 of each 50-epoch cycle.
    if (epoch % 50) + 1 > 47:
        checkpoint_path = os.path.join(
            RUN_DIR,
            f"{CHECKPOINT_PREFIX}_{checkpoint_idx}.pt",
        )

        print("save!", checkpoint_path)
        torch.save(model.state_dict(), checkpoint_path)

        checkpoint_idx += 1

print("Number of saved checkpoints:", checkpoint_idx)

## Ensemble evaluation

In [ ]:
def evaluate_single_checkpoint(model, loader, criterion, device):
    model.eval()

    losses = 0.0
    correct = 0
    total = 0
    probabilities = []
    truth = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            losses += criterion(outputs, targets).item()

            probabilities.append(F.softmax(outputs, dim=1).cpu())

            _, predicted = torch.max(outputs.data, 1)
            correct += predicted.eq(targets).sum().item()
            total += targets.size(0)

            truth.extend(targets.cpu().tolist())

    print(
        "\nCheckpoint eval: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n".format(
            losses / len(loader),
            correct,
            total,
            100.0 * correct / total,
        )
    )

    return torch.cat(probabilities, dim=0), truth


def run_ensemble_evaluation(run_dir, loader, device, num_models):
    ensemble_model = ResNet18().to(device)
    criterion = nn.CrossEntropyLoss()

    all_probs = []
    truth = None

    for idx in range(num_models):
        checkpoint = os.path.join(run_dir, f"{CHECKPOINT_PREFIX}_{idx}.pt")

        print("Loading:", checkpoint)
        state_dict = torch.load(checkpoint, map_location=device)
        ensemble_model.load_state_dict(state_dict)

        probs, truth = evaluate_single_checkpoint(
            ensemble_model,
            loader,
            criterion,
            device,
        )

        all_probs.append(probs)

    averaged_probs = sum(all_probs) / num_models
    predictions = torch.argmax(averaged_probs, dim=1).tolist()

    accuracy = sum(int(t == p) for t, p in zip(truth, predictions)) / float(len(truth))
    test_error = (1.0 - accuracy) * 100.0

    print("Ensemble accuracy: %.4f" % accuracy)
    print("Ensemble test error: %.2f%%" % test_error)

    return accuracy, test_error

In [ ]:
# If EVAL_NUM_MODELS is None, use all checkpoints saved during this notebook run.
eval_num_models = EVAL_NUM_MODELS if EVAL_NUM_MODELS is not None else checkpoint_idx

if eval_num_models <= 0:
    raise RuntimeError("No checkpoints were saved, so ensemble evaluation cannot run.")

if eval_num_models > checkpoint_idx:
    raise RuntimeError(
        f"Requested {eval_num_models} checkpoints for evaluation, "
        f"but only {checkpoint_idx} were saved."
    )

print(f"Running ensemble evaluation with {eval_num_models} checkpoints...")
accuracy, test_error = run_ensemble_evaluation(
    RUN_DIR,
    test_loader,
    device,
    eval_num_models,
)

## Optional: evaluate existing checkpoints without retraining

If you already have checkpoints in `RUN_DIR`, you can skip the training cell and manually set `checkpoint_idx`.
For example, if you have 12 checkpoints:

```python
checkpoint_idx = 12
EVAL_NUM_MODELS = None
```